# Satellite–Central Cross-Correlation
Compute the cross-correlation $\xi_{\mathrm{sat,cen}}(r)$ between satellite and central galaxies from GALFORM galaxies.hdf5 outputs.

## Setup

In [ ]:
import sys
from pathlib import Path

_src = str(Path('../../src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import DEFAULT_RBINS, get_snapshot_redshift
from analysis.correlation import satellite_central_cross_correlation
from utils.matplotlib_config import setconfig

setconfig()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11

base_dir = Path('/cosma5/data/durham/dc-hick2/Tau0_Investigation/Galform_Out_Default/L800/lc16.newmg')
zero_dir = Path('/cosma5/data/durham/dc-hick2/Tau0_Investigation/Galform_Out_0tau0/L800/lc16.newmg')
inf_dir = Path('/cosma5/data/durham/dc-hick2/Tau0_Investigation/Galform_Out_1e6tau0/L800/lc16.newmg')

## Cross-correlation for a single redshift and subvolume

In [ ]:
example_snapshot = 'iz207'
example_ivol = 1
satellite_stellar_mass_min = 1e9  # Msun/h
central_stellar_mass_min = 1e9
host_halo_mass_min = 1e9

cross = satellite_central_cross_correlation(
    base_dir / example_snapshot,
    example_ivol,
    rbins=DEFAULT_RBINS,
    nthreads=4,
    satellite_stellar_mass_min=satellite_stellar_mass_min,
    central_stellar_mass_min=central_stellar_mass_min,
    host_halo_mass_min=host_halo_mass_min,
)
cross_zero = satellite_central_cross_correlation(
    zero_dir / example_snapshot,
    example_ivol,
    rbins=DEFAULT_RBINS,
    nthreads=4,
    satellite_stellar_mass_min=satellite_stellar_mass_min,
    central_stellar_mass_min=central_stellar_mass_min,
    host_halo_mass_min=host_halo_mass_min,
)
cross_inf = satellite_central_cross_correlation(
    inf_dir / example_snapshot,
    example_ivol,
    rbins=DEFAULT_RBINS,
    nthreads=4,
    satellite_stellar_mass_min=satellite_stellar_mass_min,
    central_stellar_mass_min=central_stellar_mass_min,
    host_halo_mass_min=host_halo_mass_min,
)

z_val = get_snapshot_redshift(example_snapshot)

print(
    f"Loaded cross-correlation for {cross.attrs['iz']} (z={z_val}), ",
    f"\nbase case ivol={cross.attrs['ivol']}; N_sat={cross.attrs['n_sat']}, N_cen={cross.attrs['n_cen']}.",
    f"\nzero tau0 ivol={cross_zero.attrs['ivol']}; N_sat={cross_zero.attrs['n_sat']}, N_cen={cross_zero.attrs['n_cen']}.",
    f"\ninfinite tau0 ivol={cross_inf.attrs['ivol']}; N_sat={cross_inf.attrs['n_sat']}, N_cen={cross_inf.attrs['n_cen']}.",
)

In [ ]:
#base
r_base = cross['r'].to_numpy()
xi_base = cross['xi'].to_numpy()
valid_base = np.isfinite(xi_base) & (xi_base > -1)

#zero
r_zero = cross_zero['r'].to_numpy()
xi_zero = cross_zero['xi'].to_numpy()
valid_zero = np.isfinite(xi_zero) & (xi_zero > -1)

#inf
r_inf = cross_inf['r'].to_numpy()
xi_inf = cross_inf['xi'].to_numpy()
valid_inf = np.isfinite(xi_inf) & (xi_inf > -1)



fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(r_base[valid_base], xi_base[valid_base], 'o-', label='base')
ax.loglog(r_zero[valid_zero], xi_zero[valid_zero], 's-', label='zero')
ax.loglog(r_inf[valid_inf], xi_inf[valid_inf], 'd-', label='infinite')
ax.set_xlabel('Separation r [Mpc/h]')
ax.set_ylabel(r'$\xi_{\mathrm{sat,cen}}(r)$')
ax.set_title('Satellite-Central Cross-Correlation')
ax.legend()
plt.show()

## Compare Two Redshifts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

redshifts = ['iz271', 'iz207']

for snapshot in redshifts:
    z_val = get_snapshot_redshift(snapshot)
    z_label = f"{z_val:.2f}" if z_val is not None else "n/a"

    res_base = satellite_central_cross_correlation(
        base_dir / snapshot,
        example_ivol,
        rbins=DEFAULT_RBINS,
        nthreads=4,
        satellite_stellar_mass_min=satellite_stellar_mass_min,
        central_stellar_mass_min=central_stellar_mass_min,
        host_halo_mass_min=host_halo_mass_min,
    )
    res_zero = satellite_central_cross_correlation(
        zero_dir / snapshot,
        example_ivol,
        rbins=DEFAULT_RBINS,
        nthreads=4,
        satellite_stellar_mass_min=satellite_stellar_mass_min,
        central_stellar_mass_min=central_stellar_mass_min,
        host_halo_mass_min=host_halo_mass_min,
    )
    res_inf = satellite_central_cross_correlation(
        inf_dir / snapshot,
        example_ivol,
        rbins=DEFAULT_RBINS,
        nthreads=4,
        satellite_stellar_mass_min=satellite_stellar_mass_min,
        central_stellar_mass_min=central_stellar_mass_min,
        host_halo_mass_min=host_halo_mass_min,
    )

    if res_base is None:
        print(f"Skipped {snapshot}: no data base")
        continue
    r_base = res_base['r'].to_numpy()
    xi_base = res_base['xi'].to_numpy()
    valid_base = np.isfinite(xi_base) & (xi_base > -1)
    if not np.any(valid_base):
        print(f"Skipped {snapshot}: no finite xi base")
        continue

    if res_zero is None:
        print(f"Skipped {snapshot}: no data zero")
        continue
    r_zero = res_zero['r'].to_numpy()
    xi_zero = res_zero['xi'].to_numpy()
    valid_zero = np.isfinite(xi_zero) & (xi_zero > -1)
    if not np.any(valid_zero):
        print(f"Skipped {snapshot}: no finite xi zero")
        continue

    if res_inf is None:
        print(f"Skipped {snapshot}: no data infinite tau0")
        continue
    r_inf = res_inf['r'].to_numpy()
    xi_inf = res_inf['xi'].to_numpy()
    valid_inf = np.isfinite(xi_inf) & (xi_inf > -1)
    if not np.any(valid_inf):
        print(f"Skipped {snapshot}: no finite xi infinite tau0")
        continue

    ax.loglog(r_base[valid_base], xi_base[valid_base], 'o-', label=f"{snapshot} (z={z_label})")
    ax.loglog(r_zero[valid_zero], xi_zero[valid_zero], 's-', label=f"{snapshot} zero tau0 (z={z_label})")
    ax.loglog(r_inf[valid_inf], xi_inf[valid_inf], 'd-', label=f"{snapshot} infinite tau0 (z={z_label})")

ax.set_xlabel('Separation r [Mpc/h]')
ax.set_ylabel(r'$\xi_{\mathrm{sat,cen}}(r)$')
ax.set_title('Satellite-Central Cross-Correlation Across Redshift')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.show()

## Stack Over Subvolumes

In [ ]:
snapshots = ['iz207', 'iz271']  # Add more if needed, e.g. ['iz82', 'iz100', 'iz120', 'iz207', 'iz271']
ivols = list(range(0, 15))
r_ref = 0.5 * (DEFAULT_RBINS[:-1] + DEFAULT_RBINS[1:])

fig, (ax, ax_ratio) = plt.subplots(
    2,
    1,
    figsize=(8, 7),
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1]},
    constrained_layout=True,
 )

for snapshot in snapshots:
    z_val = get_snapshot_redshift(snapshot)
    z_label = f"{z_val:.2f}" if z_val is not None else "n/a"

    base_stack = []
    zero_stack = []
    inf_stack = []

    for ivol in ivols:
        res_base = satellite_central_cross_correlation(
            base_dir / snapshot,
            ivol,
            rbins=DEFAULT_RBINS,
            nthreads=4,
            satellite_stellar_mass_min=satellite_stellar_mass_min,
            central_stellar_mass_min=central_stellar_mass_min,
            host_halo_mass_min=host_halo_mass_min,
        )
        res_zero = satellite_central_cross_correlation(
            zero_dir / snapshot,
            ivol,
            rbins=DEFAULT_RBINS,
            nthreads=4,
            satellite_stellar_mass_min=satellite_stellar_mass_min,
            central_stellar_mass_min=central_stellar_mass_min,
            host_halo_mass_min=host_halo_mass_min,
        )
        res_inf = satellite_central_cross_correlation(
            inf_dir / snapshot,
            ivol,
            rbins=DEFAULT_RBINS,
            nthreads=4,
            satellite_stellar_mass_min=satellite_stellar_mass_min,
            central_stellar_mass_min=central_stellar_mass_min,
            host_halo_mass_min=host_halo_mass_min,
        )

        if (res_base is None) or (res_zero is None) or (res_inf is None):
            continue

        base_stack.append(res_base['xi'].to_numpy())
        zero_stack.append(res_zero['xi'].to_numpy())
        inf_stack.append(res_inf['xi'].to_numpy())

    n_used = len(base_stack)
    if n_used == 0:
        print(f"Skipped {snapshot}: no ivols had all three runs available")
        continue

    base_stack = np.asarray(base_stack, dtype=float)
    zero_stack = np.asarray(zero_stack, dtype=float)
    inf_stack = np.asarray(inf_stack, dtype=float)

    xi_mean_base = np.nanmean(base_stack, axis=0)
    xi_mean_zero = np.nanmean(zero_stack, axis=0)
    xi_mean_inf = np.nanmean(inf_stack, axis=0)

    valid_base = np.isfinite(xi_mean_base) & (xi_mean_base > -1) & np.isfinite(r_ref)
    valid_zero = np.isfinite(xi_mean_zero) & (xi_mean_zero > -1) & np.isfinite(r_ref)
    valid_inf = np.isfinite(xi_mean_inf) & (xi_mean_inf > -1) & np.isfinite(r_ref)

    if np.any(valid_base):
        ax.loglog(r_ref[valid_base], xi_mean_base[valid_base], 'o-', label=f"{snapshot} base (z={z_label}, n={n_used})")
    if np.any(valid_zero):
        ax.loglog(r_ref[valid_zero], xi_mean_zero[valid_zero], 's-', label=f"{snapshot} zero (z={z_label}, n={n_used})")
    if np.any(valid_inf):
        ax.loglog(r_ref[valid_inf], xi_mean_inf[valid_inf], 'd-', label=f"{snapshot} infinite (z={z_label}, n={n_used})")

    ratio_mask_zero = valid_base & valid_zero & (xi_mean_base > 0)
    if np.any(ratio_mask_zero):
        ratio_zero = xi_mean_zero[ratio_mask_zero] / xi_mean_base[ratio_mask_zero]
        ax_ratio.semilogx(
            r_ref[ratio_mask_zero],
            ratio_zero,
            '--',
            label=f"{snapshot} zero/base",
            alpha=0.8,
        )

    ratio_mask_inf = valid_base & valid_inf & (xi_mean_base > 0)
    if np.any(ratio_mask_inf):
        ratio_inf = xi_mean_inf[ratio_mask_inf] / xi_mean_base[ratio_mask_inf]
        ax_ratio.semilogx(
            r_ref[ratio_mask_inf],
            ratio_inf,
            '-.',
            label=f"{snapshot} infinite/base",
            alpha=0.8,
        )

ax.set_ylabel(r'$\xi_{\mathrm{sat,cen}}(r)$')
ax.set_title('Satellite-Central Cross-Correlation (stacked over ivols)')
ax.legend(ncol=1, fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3, which='both')

ax_ratio.set_xlabel('Separation r [Mpc/h]')
ax_ratio.set_ylabel('Ratio to base')
ax_ratio.axhline(1.0, color='0.5', lw=1, ls=':')
ax_ratio.legend(fontsize=8, loc='upper right')
ax_ratio.grid(True, alpha=0.3, which='both')
plt.show()

## Extended Redshift Panel
Use the same stacked method over a wider redshift set and plot each snapshot side by side.

In [ ]:
def stack_snapshot(snapshot, ivols, sat_min, cen_min, host_min, nthreads=4):
    model_dirs = {'base': base_dir, 'zero': zero_dir, 'infinite': inf_dir}
    r_ref = None
    stacks = {'base': [], 'zero': [], 'infinite': []}

    for ivol in ivols:
        runs = {}
        ok = True
        for model, model_root in model_dirs.items():
            res = satellite_central_cross_correlation(
                model_root / snapshot,
                ivol,
                rbins=DEFAULT_RBINS,
                nthreads=nthreads,
                satellite_stellar_mass_min=sat_min,
                central_stellar_mass_min=cen_min,
                host_halo_mass_min=host_min,
            )
            if res is None:
                ok = False
                break
            runs[model] = res
            if r_ref is None:
                r_ref = res['r'].to_numpy()
        if not ok:
            continue

        for model in model_dirs:
            stacks[model].append(runs[model]['xi'].to_numpy())

    out = {'r': r_ref, 'n_used': len(stacks['base'])}
    for model in stacks:
        if len(stacks[model]) == 0:
            out[model] = None
            continue
        arr = np.asarray(stacks[model], dtype=float)
        out[model] = {
            'stack': arr,
            'mean': np.nanmean(arr, axis=0),
            'sem': np.nanstd(arr, axis=0) / np.sqrt(arr.shape[0]),
        }
    return out

In [ ]:
snapshots_wide = ['iz82', 'iz100', 'iz120', 'iz207', 'iz271']
ivols_wide = list(range(0, 15))

stacked_wide = {}
for snapshot in snapshots_wide:
    s = stack_snapshot(
        snapshot,
        ivols_wide,
        satellite_stellar_mass_min,
        central_stellar_mass_min,
        host_halo_mass_min,
        nthreads=4,
    )
    stacked_wide[snapshot] = s
    print(f"{snapshot}: used {s['n_used']} ivols")

In [ ]:
valid_wide = [s for s in snapshots_wide if stacked_wide[s]['n_used'] > 0 and stacked_wide[s]['base'] is not None]
ncols = 2
nrows = int(np.ceil(len(valid_wide) / ncols)) if len(valid_wide) else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(11, 4.2 * nrows), squeeze=False, sharex=True, sharey=True)
axes = axes.flatten()

for i, snapshot in enumerate(valid_wide):
    ax = axes[i]
    s = stacked_wide[snapshot]
    r = s['r']
    z = get_snapshot_redshift(snapshot)
    z_text = f"{z:.2f}" if z is not None else 'n/a'

    for model, marker in [('base', 'o-'), ('zero', 's-'), ('infinite', 'd-')]:
        mean = s[model]['mean']
        sem = s[model]['sem']
        m = np.isfinite(mean) & np.isfinite(r) & (mean > -1)
        if not np.any(m):
            continue
        ax.loglog(r[m], mean[m], marker, label=model)
        lo = np.clip(mean - sem, -0.999, np.inf)
        hi = mean + sem
        b = m & np.isfinite(lo) & np.isfinite(hi)
        if np.any(b):
            ax.fill_between(r[b], lo[b], hi[b], alpha=0.15)

    ax.set_title(f"{snapshot} (z={z_text}, n={s['n_used']})")
    ax.set_xlabel('Separation r [Mpc/h]')
    ax.set_ylabel(r'$\xi_{\mathrm{sat,cen}}(r)$')
    ax.grid(True, alpha=0.25, which='both')
    ax.legend(fontsize=9)

for j in range(len(valid_wide), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Model Diagnostics
Use model/base ratios, bootstrap ranges, and an integrated summary to compare behavior cleanly.

In [ ]:
def safe_ratio(num, den):
    out = np.full_like(num, np.nan, dtype=float)
    m = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[m] = num[m] / den[m]
    return out

def normalize_shape(xi):
    y = 1.0 + np.asarray(xi, dtype=float)
    y = np.where(np.isfinite(y) & (y > 0), y, np.nan)
    med = np.nanmedian(y)
    if not np.isfinite(med) or med <= 0:
        return np.full_like(y, np.nan)
    return y / med

def bootstrap_ratio(base_stack, alt_stack, nboot=300, seed=11):
    rng = np.random.default_rng(seed)
    n = min(len(base_stack), len(alt_stack))
    if n == 0:
        return None, None, None
    idx = np.arange(n)
    samples = []
    for _ in range(nboot):
        pick = rng.choice(idx, size=n, replace=True)
        b = np.nanmean(base_stack[pick], axis=0)
        a = np.nanmean(alt_stack[pick], axis=0)
        samples.append(safe_ratio(a, b))
    samples = np.asarray(samples)
    return (
        np.nanmedian(samples, axis=0),
        np.nanpercentile(samples, 16, axis=0),
        np.nanpercentile(samples, 84, axis=0),
    )

def integrated_xi(r, xi, rmin=0.05, rmax=10.0):
    m = np.isfinite(r) & np.isfinite(xi) & (xi > -1) & (r >= rmin) & (r <= rmax)
    if np.count_nonzero(m) < 3:
        return np.nan
    return np.trapezoid(xi[m], np.log(r[m]))

In [ ]:
valid_diag = [s for s in snapshots_wide if stacked_wide[s]['n_used'] > 0 and stacked_wide[s]['base'] is not None]
summary_rows = []

for snapshot in valid_diag:
    s = stacked_wide[snapshot]
    r = s['r']
    base_stack = s['base']['stack']
    zero_stack = s['zero']['stack']
    inf_stack = s['infinite']['stack']

    med_zero, lo_zero, hi_zero = bootstrap_ratio(base_stack, zero_stack, nboot=300, seed=11)
    med_inf, lo_inf, hi_inf = bootstrap_ratio(base_stack, inf_stack, nboot=300, seed=17)

    fig, ax = plt.subplots(figsize=(8, 4.8))
    m0 = np.isfinite(med_zero) & np.isfinite(r)
    mi = np.isfinite(med_inf) & np.isfinite(r)

    if np.any(m0):
        ax.semilogx(r[m0], med_zero[m0], 's-', label='zero/base')
        ax.fill_between(r[m0], lo_zero[m0], hi_zero[m0], alpha=0.2)
    if np.any(mi):
        ax.semilogx(r[mi], med_inf[mi], 'd-', label='infinite/base')
        ax.fill_between(r[mi], lo_inf[mi], hi_inf[mi], alpha=0.2)

    ax.axhline(1.0, color='0.4', lw=1, ls=':')
    ax.set_xlabel('Separation r [Mpc/h]')
    ax.set_ylabel(r'$\xi_{\mathrm{model}}/\xi_{\mathrm{base}}$')
    ax.set_title(f'Ratios with bootstrap range: {snapshot}')
    ax.grid(True, alpha=0.25, which='both')
    ax.legend()
    plt.show()

    windows = {
        'small_r(<0.5)': (r >= 0.05) & (r < 0.5),
        'mid_r(0.5-2)': (r >= 0.5) & (r < 2.0),
        'large_r(2-10)': (r >= 2.0) & (r < 10.0),
    }

    mean_base = np.nanmean(base_stack, axis=0)
    mean_zero = np.nanmean(zero_stack, axis=0)
    mean_inf = np.nanmean(inf_stack, axis=0)
    ratio_zero = safe_ratio(mean_zero, mean_base)
    ratio_inf = safe_ratio(mean_inf, mean_base)

    for name, m in windows.items():
        summary_rows.append({
            'snapshot': snapshot,
            'model_ratio': 'zero/base',
            'scale_window': name,
            'mean_ratio': np.nanmean(ratio_zero[m]) if np.any(m) else np.nan,
        })
        summary_rows.append({
            'snapshot': snapshot,
            'model_ratio': 'infinite/base',
            'scale_window': name,
            'mean_ratio': np.nanmean(ratio_inf[m]) if np.any(m) else np.nan,
        })

ratio_summary_df = pd.DataFrame(summary_rows).sort_values(['snapshot', 'model_ratio', 'scale_window'])
ratio_summary_df

In [ ]:
snapshot = 'iz207' if 'iz207' in valid_diag else valid_diag[0]
s = stacked_wide[snapshot]
r = s['r']

shape_base = normalize_shape(s['base']['mean'])
shape_zero = normalize_shape(s['zero']['mean'])
shape_inf = normalize_shape(s['infinite']['mean'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, constrained_layout=True)

for model, marker in [('base', 'o-'), ('zero', 's-'), ('infinite', 'd-')]:
    y = s[model]['mean']
    m = np.isfinite(y) & np.isfinite(r) & (y > -1)
    if np.any(m):
        ax1.loglog(r[m], y[m], marker, label=model)

for y, label, marker in [
    (shape_base, 'base shape', 'o-'),
    (shape_zero, 'zero shape', 's-'),
    (shape_inf, 'infinite shape', 'd-'),
]:
    m = np.isfinite(y) & np.isfinite(r) & (y > 0)
    if np.any(m):
        ax2.semilogx(r[m], y[m], marker, label=label)

ax1.set_ylabel(r'$\xi_{\mathrm{sat,cen}}(r)$')
ax1.set_title(f'Shape and amplitude check ({snapshot})')
ax1.grid(True, alpha=0.25, which='both')
ax1.legend()

ax2.set_xlabel('Separation r [Mpc/h]')
ax2.set_ylabel(r'normalized $(1+\xi)$')
ax2.axhline(1.0, color='0.4', lw=1, ls=':')
ax2.grid(True, alpha=0.25, which='both')
ax2.legend(fontsize=9)
plt.show()

rows = []
for snapshot in valid_diag:
    s = stacked_wide[snapshot]
    r = s['r']
    z = get_snapshot_redshift(snapshot)
    j_base = integrated_xi(r, s['base']['mean'])
    j_zero = integrated_xi(r, s['zero']['mean'])
    j_inf = integrated_xi(r, s['infinite']['mean'])

    rows.append({
        'snapshot': snapshot,
        'z': z,
        'J_base': j_base,
        'J_zero': j_zero,
        'J_infinite': j_inf,
        'ratio_zero_base': j_zero / j_base if np.isfinite(j_base) and j_base != 0 else np.nan,
        'ratio_inf_base': j_inf / j_base if np.isfinite(j_base) and j_base != 0 else np.nan,
        'n_used': s['n_used'],
    })

evo_df = pd.DataFrame(rows).sort_values('z')
evo_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(evo_df['z'], evo_df['ratio_zero_base'], 's-', label='zero/base')
ax.plot(evo_df['z'], evo_df['ratio_inf_base'], 'd-', label='infinite/base')
ax.axhline(1.0, color='0.4', ls=':', lw=1)
ax.set_xlabel('Redshift z')
ax.set_ylabel(r'$J_{\\mathrm{model}}/J_{\\mathrm{base}}$')
ax.set_title('Integrated trend with redshift')
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Selection Scan
Keep cut-sensitivity plots in the same notebook, since they are directly comparable to the cross-correlation diagnostics.

In [ ]:
snapshot_scan = 'iz207'
ivols_scan = list(range(0, 10))
sat_cuts = [1e9, 3e9, 1e10]
host_cuts = [1e11, 3e11, 1e12]

rows_scan = []
for sat_cut in sat_cuts:
    for host_cut in host_cuts:
        s = stack_snapshot(snapshot_scan, ivols_scan, sat_cut, 1e9, host_cut, nthreads=4)
        if s['n_used'] == 0 or s['base'] is None:
            continue

        r = s['r']
        small = (r >= 0.05) & (r < 0.5)
        if not np.any(small):
            continue

        base_mean = np.nanmean(s['base']['mean'][small])
        zero_mean = np.nanmean(s['zero']['mean'][small])
        inf_mean = np.nanmean(s['infinite']['mean'][small])

        rows_scan.append({
            'snapshot': snapshot_scan,
            'sat_min': sat_cut,
            'host_min': host_cut,
            'n_used': s['n_used'],
            'xi_small_base': base_mean,
            'xi_small_zero': zero_mean,
            'xi_small_infinite': inf_mean,
            'ratio_zero_base': zero_mean / base_mean if np.isfinite(base_mean) and base_mean != 0 else np.nan,
            'ratio_infinite_base': inf_mean / base_mean if np.isfinite(base_mean) and base_mean != 0 else np.nan,
        })

scan_df = pd.DataFrame(rows_scan).sort_values(['sat_min', 'host_min'])
scan_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

for sat_cut in sat_cuts:
    sub = scan_df[scan_df['sat_min'] == sat_cut]
    if len(sub) == 0:
        continue
    x = np.log10(sub['host_min'].to_numpy())
    axes[0].plot(x, sub['ratio_zero_base'].to_numpy(), 'o-', label=f'sat > {sat_cut:.0e}')
    axes[1].plot(x, sub['ratio_infinite_base'].to_numpy(), 'd-', label=f'sat > {sat_cut:.0e}')

axes[0].axhline(1.0, color='0.4', ls=':', lw=1)
axes[1].axhline(1.0, color='0.4', ls=':', lw=1)

axes[0].set_title('zero/base at small scales')
axes[1].set_title('infinite/base at small scales')
axes[0].set_ylabel('ratio')
axes[1].set_ylabel('ratio')
axes[0].set_xlabel(r'$\log_{10}(M_{\mathrm{halo,min}}/[M_\odot/h])$')
axes[1].set_xlabel(r'$\log_{10}(M_{\mathrm{halo,min}}/[M_\odot/h])$')
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.show()

In [ ]:
out_dir = Path('data/convergence/dynamical_friction_extensions')
out_dir.mkdir(parents=True, exist_ok=True)
scan_df.to_csv(out_dir / 'selection_scan_small_scale.csv', index=False)
print('Wrote', (out_dir / 'selection_scan_small_scale.csv').resolve())